# Full pipeline demo
text -> chunks -> Transformer -> Diffusion -> GAN vocoder -> audio. Needs all five `*_best.pt` checkpoints. Run the setup cell first.

In [1]:
!pip install -q librosa soundfile num2words faster-whisper


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: C:\Users\INTEL\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [2]:
import sys, torch
ROOT = r"C:\Users\INTEL\Downloads\voice_cloning_v2_1\voice_cloning_v2_1"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

GPU: NVIDIA GeForce RTX 4070


In [3]:
from IPython.display import Audio, display
from vc_lib.infer import VoicePipeline, evaluate
from vc_lib.common import SR

pipe = VoicePipeline(ROOT, diffusion_steps=50, cfg=1.5)
pipe.speakers()

{'train': ['p225',
  'p226',
  'p227',
  'p228',
  'p229',
  'p230',
  'p231',
  'p232',
  'p233',
  'p234',
  'p236',
  'p237',
  'p238',
  'p239',
  'p240',
  'p241',
  'p243',
  'p244',
  'p245',
  'p246',
  'p247',
  'p248',
  'p249',
  'p250',
  'p251',
  'p252',
  'p253',
  'p254',
  'p255',
  'p256',
  'p257',
  'p258',
  'p259',
  'p260',
  'p261',
  'p262',
  'p267',
  'p268',
  'p269',
  'p270',
  'p271',
  'p272',
  'p273',
  'p274',
  'p275',
  'p276',
  'p277',
  'p278',
  'p279',
  'p280',
  'p281',
  'p282',
  'p283',
  'p284',
  'p285',
  'p286',
  'p287',
  'p288',
  'p292',
  'p293',
  'p294',
  'p295',
  'p297',
  'p298',
  'p299',
  'p300',
  'p301',
  'p302',
  'p303',
  'p304',
  'p305',
  'p306',
  'p307',
  'p308',
  'p310',
  'p311'],
 'holdout (unseen)': ['p263', 'p264', 'p265', 'p266']}

In [4]:
text = "Hello! Thanks for calling. How can I help you today?"
spk = pipe.meta['train_speakers'][0]
display(Audio(pipe.synthesize(text, speaker=spk), rate=SR))

c:\Users\INTEL\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## With vs without diffusion
If the fast version sounds as good, skip diffusion in production.

In [5]:
fast = VoicePipeline(ROOT, use_diffusion=False)
display(Audio(fast.synthesize(text, speaker=spk), rate=SR))

## Zero-shot: an unseen speaker

In [6]:
unseen = pipe.meta['holdout'][0]
display(Audio(pipe.synthesize(text, speaker=unseen), rate=SR))

## Clone your own voice
Upload 1-3 clean recordings (5-10 s total) of the target voice.

In [16]:
ref = r"C:\Users\INTEL\Downloads\Recording.wav"   # your reference recording
emb = pipe.embed_reference([ref])
display(Audio(pipe.synthesize(text, speaker_emb=emb), rate=SR))

## Evaluate
WER (Whisper) = intelligibility, lower is better. Similarity = voice match, higher is better.

In [18]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper", "pydub"], check=True)
import faster_whisper
print("faster-whisper installed in", sys.executable)

faster-whisper installed in c:\Users\INTEL\AppData\Local\Programs\Python\Python311\python.exe


In [19]:
sentences = ["Please call Stella.", "The weather is nice today.", "I will send the report tomorrow morning.",
             "Can you repeat that, please?", "Your order has been shipped."]
evaluate(pipe, sentences, speaker=spk)
evaluate(pipe, sentences, speaker=unseen)

c:\Users\INTEL\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\INTEL\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\INTEL\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning:

WER 2.33 | similarity 0.30 | 'Please call Stella.' -> 'Thank you so much for your time.'
WER 1.40 | similarity 0.50 | 'The weather is nice today.' -> 'Thank you very much for your time.'
WER 1.14 | similarity 0.50 | 'I will send the report tomorrow morning.' -> 'Thank you very much.  Thank you very much.'
WER 0.80 | similarity 0.34 | 'Can you repeat that, please?' -> 'Thank you very much.'
WER 1.20 | similarity 0.44 | 'Your order has been shipped.' -> "I'm going to get the slides."

Mean WER: 1.375 (lower is better)   Mean speaker similarity: 0.416
WER 1.33 | similarity 0.35 | 'Please call Stella.' -> 'Yes, yes, yes, yes.'
WER 1.00 | similarity 0.59 | 'The weather is nice today.' -> 'Thank you very much.'
WER 1.00 | similarity 0.63 | 'I will send the report tomorrow morning.' -> 'Thank you very much.'
WER 1.40 | similarity 0.42 | 'Can you repeat that, please?' -> 'Thank you, thank you, thank you, thank you.'
WER 1.00 | similarity 0.56 | 'Your order has been shipped.' -> 'Thank you very

([1.3333333333333333, 1.0, 1.0, 1.4, 1.0],
 [0.3501979112625122,
  0.5946436524391174,
  0.6278563737869263,
  0.41683638095855713,
  0.5645972490310669])

In [20]:
from vc_lib.common import paths, load_cache
C = load_cache(paths(ROOT))
rows = [k for k in C["idx"]["train"].tolist() if C["spks"][k] == spk][::200][:5]
evaluate(pipe, [C["texts"][k] for k in rows], speaker=spk)

WER 2.33 | similarity 0.30 | 'please call stella' -> 'Thank you so much for your time.'
WER 1.75 | similarity 0.57 | 'a hostile bid is' -> 'Thank you very much for your time.'
WER 1.33 | similarity 0.33 | 'mr hall is' -> 'Thank you for watching!'
WER 1.00 | similarity 0.39 | 'was high on the council' -> ''
WER 3.50 | similarity 0.56 | 'refugees already' -> 'Thank you so much for your time.'

Mean WER: 1.983 (lower is better)   Mean speaker similarity: 0.431


([2.3333333333333335, 1.75, 1.3333333333333333, 1.0, 3.5],
 [0.30334100127220154,
  0.5682388544082642,
  0.33349138498306274,
  0.3877321481704712,
  0.5645304322242737])

In [21]:
import numpy as np, torch, librosa
from faster_whisper import WhisperModel
from vc_lib.common import paths, load_cache, device, SR, encode_text
from vc_lib.train_diffusion import ddim_sample

C = load_cache(paths(ROOT))
k = [i for i in C["idx"]["train"].tolist() if C["spks"][i] == spk][100]
text = C["texts"][k]
wh = WhisperModel("small", device="cpu", compute_type="int8")

def hear(y):
    segs, _ = wh.transcribe(librosa.resample(y, orig_sr=SR, target_sr=16000), language="en", beam_size=1)
    return " ".join(s.text for s in segs).strip()

def stats(name, m=None, y=None):
    if m is not None:
        print(f"{name:10s} mel mean {m.mean():.3f}  max {m.max():.3f}")
    if y is not None:
        print(f"{name:10s} wav peak {np.abs(y).max():.3f}  rms {np.sqrt((y**2).mean()):.4f}  heard: {hear(y)!r}")

print("target text:", repr(text))
with torch.no_grad():
    gt = C["mels"][k].float()[None, None].to(device)
    y_gt = pipe.G(gt).squeeze().cpu().numpy()
    stats("truth-mel", m=gt.cpu().numpy(), y=y_gt)

    ids, mask = encode_text(text, pipe.meta["char2id"])
    ids = torch.from_numpy(ids)[None].to(device)
    mask = torch.from_numpy(mask)[None].to(device)
    e = pipe.speaker_mean[spk].to(device)[None]
    coarse = pipe.tts(ids, mask, e)
    refined = ddim_sample(pipe.diff, pipe.sched, coarse, e, 50, 1.5, seed=0)
    y_c = pipe.G(coarse).squeeze().cpu().numpy()
    y_r = pipe.G(refined).squeeze().cpu().numpy()
    stats("coarse", m=coarse.cpu().numpy(), y=y_c)
    stats("refined", m=refined.cpu().numpy(), y=y_r)

target text: 'as a bridge over'
truth-mel  mel mean 0.358  max 1.000
truth-mel  wav peak 0.450  rms 0.0736  heard: 'as a bridge over.'
coarse     mel mean 0.396  max 0.835
coarse     wav peak 0.129  rms 0.0338  heard: ''
refined    mel mean 0.433  max 1.000
refined    wav peak 0.371  rms 0.0711  heard: 'Thank you.'


In [22]:
for cfg in [1.0, 1.2, 1.5]:
    with torch.no_grad():
        r = ddim_sample(pipe.diff, pipe.sched, coarse, e, 50, cfg, seed=0)
        y = pipe.G(r).squeeze().cpu().numpy()
    stats(f"cfg {cfg}", m=r.cpu().numpy(), y=y)

cfg 1.0    mel mean 0.414  max 0.999
cfg 1.0    wav peak 0.373  rms 0.0552  heard: 'Thank you.'
cfg 1.2    mel mean 0.423  max 1.000
cfg 1.2    wav peak 0.373  rms 0.0620  heard: 'Thank you.'
cfg 1.5    mel mean 0.433  max 1.000
cfg 1.5    wav peak 0.371  rms 0.0711  heard: 'Thank you.'


In [23]:
with torch.no_grad():
    r = ddim_sample(pipe.diff, pipe.sched, coarse, e, 50, 1.0, seed=0)
    sil = coarse < 0.05   # frames the Transformer says are silent
    print("refined in silent frames:", round(float(r[sil].mean()), 3),
          "| truth in its silent frames:", round(float(gt[gt < 0.05].mean()), 3) if (gt < 0.05).any() else "n/a")
    g = r.clone()
    g[sil] = 0.0                       # force silence where the Transformer has no speech
    y = pipe.G(g).squeeze().cpu().numpy()
    stats("gated", m=g.cpu().numpy(), y=y)

refined in silent frames: 0.166 | truth in its silent frames: 0.005
gated      mel mean 0.390  max 0.999
gated      wav peak 0.373  rms 0.0551  heard: 'Thank you.'


In [24]:
with torch.no_grad():
    y = pipe.G(coarse).squeeze().cpu().numpy()    # coarse mel, no diffusion
    stats("no-diff", m=coarse.cpu().numpy(), y=y)
    t = pipe.G(gt).squeeze().cpu().numpy()        # already known: truth-mel

no-diff    mel mean 0.396  max 0.835
no-diff    wav peak 0.129  rms 0.0338  heard: ''


In [25]:
with torch.no_grad():
    m = coarse.mean()
    for g in [1.0, 1.5, 2.0]:
        c = (m + g * (coarse - m)).clamp(0, 1)
        stats(f"stretch {g}", m=c.cpu().numpy(), y=pipe.G(c).squeeze().cpu().numpy())

stretch 1.0 mel mean 0.396  max 0.835
stretch 1.0 wav peak 0.129  rms 0.0338  heard: ''
stretch 1.5 mel mean 0.425  max 1.000
stretch 1.5 wav peak 0.373  rms 0.1107  heard: 'I'
stretch 2.0 mel mean 0.450  max 1.000
stretch 2.0 wav peak 0.460  rms 0.1471  heard: ''


In [26]:
import torch
from vc_lib.common import paths, load_cache, encode_text, device

texts = ["as a bridge over", "please call stella", "the weather is nice today", "refugees already"]
e = pipe.speaker_mean[spk].to(device)[None]
outs = []
with torch.no_grad():
    for t in texts:
        ids, mask = encode_text(t, pipe.meta["char2id"])
        out = pipe.tts(torch.from_numpy(ids)[None].to(device), torch.from_numpy(mask)[None].to(device), e)
        outs.append(out.squeeze().cpu())

for a in range(len(texts)):
    for b in range(a + 1, len(texts)):
        print(f"{texts[a]!r} vs {texts[b]!r}: diff {(outs[a] - outs[b]).abs().mean():.4f}")

C = load_cache(paths(ROOT))
m = C["mels"][C["idx"]["train"][:100]].float()
print("typical diff between two different real mels:", round((m[0::2] - m[1::2]).abs().mean().item(), 4))

'as a bridge over' vs 'please call stella': diff 0.1246
'as a bridge over' vs 'the weather is nice today': diff 0.0678
'as a bridge over' vs 'refugees already': diff 0.1174
'please call stella' vs 'the weather is nice today': diff 0.0908
'please call stella' vs 'refugees already': diff 0.0598
'the weather is nice today' vs 'refugees already': diff 0.1048
typical diff between two different real mels: 0.2307


## Save audio to Drive

In [ ]:
import soundfile as sf
sf.write(f"{ROOT}/demo_output.wav", pipe.synthesize(text, speaker=spk), SR)